# Piloto — Explicabilidad (Attention Rollout) — moondream2 — Benchmark Sintético 2D

Implementa **attention rollout** (Abnar & Zuidema, 2020) sobre el codificador visual (ViT) de `moondream2`, para ver en qué parches de la imagen se concentra la atención del modelo al responder cada uno de los 4 tipos de pregunta del benchmark sintético 2D (`conteo_general`, `conteo_forma`, `conteo_color`, `conteo_combinado`).

**Por qué attention rollout y no Grad-CAM:** los cuatro modelos evaluados en este trabajo usan codificadores de imagen tipo Vision Transformer (ViT), no redes convolucionales — Grad-CAM depende de mapas de activación convolucionales y no es directamente aplicable a un ViT. Attention rollout, en cambio, opera directamente sobre las matrices de atención que un ViT ya calcula en cada capa, por lo que no requiere adaptación estructural del modelo.

**Cómo usar este notebook:**
1. Sube a Colab el `.zip` del benchmark sintético 2D generado con `2D.py` (mismo formato usado en el piloto de evaluación).
2. Corre las celdas en orden. La celda 6 (diagnóstico de módulos) imprime los nombres reales de los submódulos de atención del modelo — **revísala antes de correr la celda de captura de atención**, porque `trust_remote_code=True` significa que la implementación interna de moondream2 puede cambiar entre revisiones del checkpoint, y los nombres de módulo capturados ahí son los que hay que ajustar en `ATTN_MODULE_NAME_PATTERN` si no calzan.
3. El notebook guarda, por cada imagen procesada, el heatmap de rollout superpuesto sobre la imagen original en `OUTPUT_DIR`, más un CSV resumen (`rollout_resumen.csv`) con métricas simples de concentración de atención por categoría de pregunta.
4. **Este notebook es exploratorio, a diferencia de los pilotos de evaluación (que ya corrieron sin cambios).** Como no hay GPU disponible fuera de Colab en este proyecto, no fue posible ejecutarlo ni validar sus resultados antes de entregarlo — la Memoria lo referencia como diseño de experimento pendiente de ejecución (sección 3.4).

## 1. Configuración de la corrida

In [ ]:
MODEL_ID = "vikhyatk/moondream2"
MODEL_REVISION = "2025-06-21"

ZIP_PATH = "/content/benchmark_2d_sintetico_lotes.zip"  # sube este archivo (generado con 2D.py) al panel de archivos de Colab antes de correr esta celda
EXTRACT_DIR = "/content/dataset_2d_sintetico"

OUTPUT_DIR = "/content/attention_rollout_moondream2"
N_IMAGENES_POR_TIPO = 3   # cuántas imágenes de ejemplo procesar por cada uno de los 4 tipos de pregunta
RANDOM_SEED = 42

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Heatmaps y CSV se guardarán en: {OUTPUT_DIR}")


## 2. Extracción del dataset

Mismo `.zip` (`dataset.json` + carpeta `images/`) usado en el piloto de evaluación.

In [ ]:
import zipfile, json, os

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zipf:
    zipf.extractall(EXTRACT_DIR)

IMAGES_DIR = os.path.join(EXTRACT_DIR, "images")
DATASET_JSON_PATH = os.path.join(EXTRACT_DIR, "dataset.json")

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(f"Extraído en: {EXTRACT_DIR}")
print(f"Preguntas en dataset.json: {len(qa_data)}")
tipos = sorted(set(item["type"] for item in qa_data))
print(f"Tipos de pregunta encontrados: {tipos}")

### Fix de compatibilidad: `transformers` 5.x rompe modelos con `trust_remote_code`

Mismo problema documentado en los pilotos de evaluación de InternVL2-1B y moondream2: Colab trae por defecto `transformers>=5`, que exige un atributo (`all_tied_weights_keys`) que el código de moondream2 (`trust_remote_code=True`, escrito para `transformers` 4.x) no define — produce `AttributeError: '...' object has no attribute 'all_tied_weights_keys'`.

**Corre la celda de abajo una sola vez y luego reinicia el entorno de ejecución** (Entorno de ejecución → Reiniciar sesión) antes de seguir.

In [ ]:
!pip install -q "transformers==4.46.3"
print("transformers fijado a 4.46.3 -- REINICIA el entorno de ejecucion antes de continuar.")

## 3. Carga del modelo (con atención "eager" para poder capturarla)

Se fuerza `attn_implementation="eager"` al cargar el modelo: con la implementación "eager", el mecanismo de atención se calcula explícitamente vía softmax en Python (en vez de un kernel fusionado tipo SDPA/FlashAttention), lo que permite capturar la matriz de atención de cada capa con un *forward hook* estándar de PyTorch. Sin este flag, las matrices de atención normalmente no quedan expuestas.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device}")
if device == "cpu":
    print("ADVERTENCIA: no se detectó GPU. Activa GPU en 'Entorno de ejecución > Cambiar tipo de entorno de ejecución'.")

model_kwargs = dict(
    trust_remote_code=True,
    revision=MODEL_REVISION,
    attn_implementation="eager",   # necesario para poder capturar las matrices de atención
)
if device == "cuda":
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs).to(device).eval()
print("Modelo cargado.")

## 4. Diagnóstico: por qué los forward hooks no funcionan en moondream2 (y qué hacer en su lugar)

**Resultado de la primera corrida de este notebook (v1):** 0/12 imágenes procesadas -- ninguno de los 153 módulos que calzaban con el patrón `"attn"` entregó una matriz de atención capturable. Revisando el código fuente real de moondream2 (`vision.py` / `layers.py`, revisión `2025-06-21`) se confirmó por qué, por dos razones independientes:

1. **Los módulos `*.attn` son `ModuleDict`, no capas con lógica propia.** Son solo contenedores de pesos (`qkv`, `proj`); nunca se "llaman" directamente como módulo, así que su forward hook jamás se dispara.
2. **La atención real se calcula con `torch.nn.functional.scaled_dot_product_attention(q, k, v)`** (kernel fusionado), dentro de una función de Python suelta (`attn()` en `layers.py`), no dentro de un `nn.Module`. Un forward hook solo puede engancharse a la llamada de un `nn.Module` -- y `scaled_dot_product_attention` además no devuelve la matriz de pesos aunque se pudiera enganchar: el kernel fusionado nunca la materializa.

En otras palabras: los hooks sobre `.qkv`/`.proj` sí capturaban tensores reales (de ahí los avisos "salida de tipo Tensor", no un error), pero esos tensores son las proyecciones lineales de entrada/salida, no los pesos de atención -- la matriz `softmax(QK^T/√d)` en sí nunca existe como salida de un módulo visible para `named_modules()`.

**Solución: monkey-patch de la función `attn()` de `vision.py`.** En vez de enganchar un hook, se reemplaza en tiempo de ejecución la función `attn` que usa el codificador visual por una reimplementación manual -- idéntica en proyecciones y reshape, pero que calcula `softmax(QK^T/√d)` explícitamente en vez de llamar al kernel fusionado, y guarda esa matriz antes de continuar. El decodificador de texto define su **propia** función `attn()` (en `text.py`, no importada de `layers.py`), así que este patch no lo toca -- no hace falta filtrar manualmente capas de visión vs. texto como se intentaba en la v1.

La celda de abajo localiza los módulos dinámicos (`vision.py` y `layers.py`) cargados por `trust_remote_code=True` dentro de `sys.modules`, e imprime sus nombres para que puedas confirmar que son los correctos antes de aplicar el patch en la sección 5.


In [ ]:
import sys

def localizar_modulos_moondream():
    """Busca en sys.modules los módulos dinámicos de moondream2 cargados por
    trust_remote_code=True: vision.py (define vision_encoder y usa attn importada
    de layers.py) y layers.py (define attn y linear, usados por vision.py)."""
    vision_mod, layers_mod = None, None
    for modname, mod in list(sys.modules.items()):
        if mod is None or "moondream" not in modname.lower():
            continue
        if modname.endswith(".vision") and hasattr(mod, "vision_encoder") and hasattr(mod, "attn"):
            vision_mod = mod
        if modname.endswith(".layers") and hasattr(mod, "attn") and hasattr(mod, "linear"):
            layers_mod = mod
    return vision_mod, layers_mod

vision_mod, layers_mod = localizar_modulos_moondream()

print(f"vision.py encontrado: {vision_mod.__name__ if vision_mod else 'NO ENCONTRADO'}")
print(f"layers.py encontrado: {layers_mod.__name__ if layers_mod else 'NO ENCONTRADO'}")

assert vision_mod is not None and layers_mod is not None, (
    "No se encontraron los módulos dinámicos esperados -- es posible que esta revisión "
    "de moondream2 haya cambiado de estructura interna respecto a 2025-06-21. "
    "Revisa manualmente sys.modules o el código fuente del checkpoint en HuggingFace."
)


## 5. Captura de atención: monkey-patch de `attn()` (reemplaza los forward hooks)

Se guarda la función `attn` original de `vision.py` (por si se quiere revertir, sección 10) y se reemplaza por una versión instrumentada: mismas proyecciones `qkv`/`proj` y mismo reshape a `[batch, heads, tokens, head_dim]` que el original, pero calculando `softmax(QK^T/√d)` manualmente en vez de delegar en `F.scaled_dot_product_attention`, y agregando esa matriz a `captured_attentions` antes de devolver el resultado de la atención (matemáticamente equivalente al original, solo que ya no usa el kernel fusionado).

Como `vision.py` hizo `from .layers import attn` al cargarse, el nombre `attn` que usa `vision_encoder()` en tiempo de ejecución vive en el namespace de `vision.py` (`vision_mod.attn`), no en el de `layers.py` -- por eso el patch se aplica ahí. `text.py` define su propia función `attn()` (no la importa de `layers.py`), así que el decodificador de texto sigue usando el kernel fusionado sin modificar.


In [ ]:
import torch.nn.functional as F

captured_attentions = []  # se llena en cada forward del encoder visual; lista de tensores [heads, tokens, tokens]
_original_vision_attn = vision_mod.attn  # para poder revertir el patch (sección 10)
CAPTURE_ENABLED = True  # poner en False para desactivar la captura sin revertir el patch

def patched_attn(x, w, n_heads):
    """Reimplementación de layers.attn(): mismas proyecciones y reshape que el original,
    pero softmax(QK^T/√d) explícito en vez de F.scaled_dot_product_attention, para poder
    guardar la matriz de atención antes de descartarla."""
    bsz, q_len, d_model = x.shape
    head_dim = d_model // n_heads
    q, k, v = [
        t.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
        for t in layers_mod.linear(x, w.qkv).chunk(3, dim=-1)
    ]
    attn_weights = (q @ k.transpose(-2, -1)) / (head_dim ** 0.5)
    attn_weights = attn_weights.softmax(dim=-1)
    if CAPTURE_ENABLED:
        captured_attentions.append(attn_weights.detach().float().cpu()[0])  # quita el batch -> [heads, tokens, tokens]
    out = attn_weights @ v
    out = out.transpose(1, 2).reshape(bsz, q_len, d_model)
    out = layers_mod.linear(out, w.proj)
    return out

vision_mod.attn = patched_attn
print("vision_mod.attn reemplazada por la versión instrumentada "
      "(el decodificador de texto no se ve afectado).")


## 6. Attention rollout (Abnar & Zuidema, 2020)

Implementación estándar del algoritmo:
1. Promediar la atención entre cabezas (heads) en cada capa.
2. Sumar la matriz identidad a cada capa promediada (modela la conexión residual: cada token también "atiende a sí mismo" fuera del mecanismo de atención) y renormalizar cada fila para que sume 1.
3. Multiplicar las matrices de todas las capas en orden (capa 1 · capa 2 · ... · capa N) para obtener el mapa de rollout acumulado, que aproxima cómo se propaga la información desde cada parche de entrada hasta la salida a través de toda la red.
4. Tomar la fila correspondiente al token de salida relevante (o, para una vista global, la atención acumulada recibida por cada parche) y proyectarla sobre la grilla de parches de la imagen.

In [ ]:
import numpy as np

def attention_rollout(attn_layers):
    """attn_layers: lista de tensores [heads, tokens, tokens], uno por capa, en orden.
    Devuelve la matriz de rollout acumulada [tokens, tokens]."""
    n_tokens = attn_layers[0].shape[-1]
    rollout = torch.eye(n_tokens)
    for attn in attn_layers:
        attn_avg = attn.mean(dim=0)                          # promedio entre heads -> [tokens, tokens]
        attn_avg = attn_avg + torch.eye(n_tokens)             # + identidad (conexión residual)
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)  # renormalizar filas
        rollout = attn_avg @ rollout                           # acumular capa por capa
    return rollout

def rollout_to_heatmap(rollout, grid_size, cls_token_offset=1):
    """Toma la fila del token [CLS] (si existe, offset=1) o el promedio de todas las filas
    si no hay CLS, y la reordena en la grilla espacial de parches [grid_size, grid_size]."""
    if cls_token_offset > 0 and rollout.shape[0] > grid_size * grid_size:
        row = rollout[0, cls_token_offset:cls_token_offset + grid_size * grid_size]
    else:
        row = rollout.mean(dim=0)[-grid_size * grid_size:]
    heatmap = row.reshape(grid_size, grid_size).numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap

print("attention_rollout() y rollout_to_heatmap() definidas.")

## 7. Selección de imágenes de ejemplo y ejecución

Se toman `N_IMAGENES_POR_TIPO` preguntas de cada uno de los 4 tipos (`conteo_general`, `conteo_forma`, `conteo_color`, `conteo_combinado`), con semilla fija, para poder comparar cualitativamente si el patrón de atención cambia según lo que la pregunta le pide al modelo mirar — que es exactamente lo que se quiere observar (Memoria, sección 3.4).

In [ ]:
import random
random.seed(RANDOM_SEED)

muestra = []
for tipo in tipos:
    candidatos = [item for item in qa_data if item["type"] == tipo]
    muestra.extend(random.sample(candidatos, min(N_IMAGENES_POR_TIPO, len(candidatos))))

print(f"Muestra total: {len(muestra)} preguntas ({N_IMAGENES_POR_TIPO} por tipo x {len(tipos)} tipos)")
for item in muestra:
    print(f"  [{item['type']:>17}] {item['image']}  ->  Q: {item['question'][:70]}...")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import csv

resumen_rows = []

for item in muestra:
    img_path = os.path.join(IMAGES_DIR, item["image"])
    imagen = Image.open(img_path).convert("RGB")

    captured_attentions.clear()

    # model.query() ejecuta vision_encoder() internamente, que ahora usa la versión
    # instrumentada de attn() (sección 5) -- no hace falta llamar al encoder por separado.
    with torch.no_grad():
        _ = model.query(imagen, item["question"])

    if len(captured_attentions) == 0:
        print(f"[AVISO] no se capturó atención para {item['image']} ({item['type']}) -- "
              f"revisar sección 4/5 y ajustar la vía de captura.")
        continue

    rollout = attention_rollout(captured_attentions)
    n_tokens = rollout.shape[0]
    grid_size = int(round((n_tokens - 1) ** 0.5)) if n_tokens > 1 else 1  # asume 1 token [CLS] + grilla cuadrada; ajustar si no aplica
    if grid_size * grid_size + 1 != n_tokens:
        grid_size = int(round(n_tokens ** 0.5))  # fallback: sin token [CLS] separado

    heatmap = rollout_to_heatmap(rollout, grid_size)
    heatmap_img = Image.fromarray((cm.jet(heatmap)[:, :, :3] * 255).astype(np.uint8)).resize(imagen.size, Image.BILINEAR)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
    axes[0].imshow(imagen); axes[0].set_title(item["image"]); axes[0].axis("off")
    axes[1].imshow(imagen); axes[1].imshow(heatmap_img, alpha=0.55); axes[1].set_title(f"Rollout -- {item['type']}"); axes[1].axis("off")
    fig.suptitle(item["question"][:90], fontsize=9)
    plt.tight_layout()
    out_name = f"rollout_{item['type']}_{item['image'].replace('.png','')}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, out_name), dpi=130)
    plt.close(fig)

    # métrica simple de concentración: fracción de la masa de atención en el 25% de parches más atendidos
    flat = np.sort(heatmap.flatten())[::-1]
    top_quarter = flat[:max(1, len(flat) // 4)]
    concentracion = top_quarter.sum() / (flat.sum() + 1e-8)

    resumen_rows.append({
        "image": item["image"], "type": item["type"], "question": item["question"],
        "n_layers_capturadas": len(captured_attentions), "grid_size": grid_size,
        "concentracion_top25pct": round(float(concentracion), 4),
        "heatmap_file": out_name,
    })
    print(f"OK  [{item['type']:>17}] {item['image']}  ->  {out_name}  (concentración top-25%: {concentracion:.3f})")

with open(os.path.join(OUTPUT_DIR, "rollout_resumen.csv"), "w", newline="", encoding="utf-8") as f:
    if resumen_rows:
        writer = csv.DictWriter(f, fieldnames=list(resumen_rows[0].keys()))
        writer.writeheader()
        writer.writerows(resumen_rows)

print(f"\n{len(resumen_rows)}/{len(muestra)} imágenes procesadas correctamente. Resumen: rollout_resumen.csv")

## 8. Otras técnicas de explicabilidad investigadas (no implementadas)

Durante la etapa de diseño de este experimento se evaluaron, además de attention rollout, las siguientes alternativas — se documentan aquí, junto con la razón por la que no se usaron como técnica principal, porque esa comparación es parte del aporte metodológico de este trabajo (Memoria, sección 3.4):

- **Grad-CAM / Grad-CAM++**: la técnica más establecida para explicabilidad visual, pero diseñada para mapas de activación convolucionales (CNN). Los cuatro modelos evaluados en este trabajo usan codificadores ViT, sin las capas convolucionales sobre las que Grad-CAM calcula su gradiente — no es directamente aplicable sin antes adaptar la técnica a atención, lo que en la práctica converge a algo equivalente a attention rollout o a Attention Flow.
- **Attention Flow**: variante de attention rollout que modela la propagación de atención entre capas como un problema de flujo máximo en un grafo, en vez de una simple multiplicación de matrices. Es, en teoría, más precisa que attention rollout puro, pero computacionalmente más cara (resolver max-flow por cada par de tokens) y bastante más compleja de implementar correctamente; se descartó por el tiempo disponible para esta etapa del trabajo, quedando como una posible extensión futura si attention rollout resulta insuficiente.
- **Integrated Gradients**: técnica de atribución basada en gradientes, agnóstica a la arquitectura (aplicable tanto a CNN como a ViT). Se descartó principalmente porque requiere múltiples forward+backward passes por cada predicción (integración a lo largo de un camino desde una imagen base hasta la imagen real), lo que la vuelve mucho más costosa computacionalmente que attention rollout — un factor relevante dado que este trabajo ya corre sin GPU disponible fuera de Colab.
- **Atención cruzada texto-imagen** (en vez de solo la auto-atención del codificador visual): en principio más directamente interpretable para VQA, porque mostraría a qué parches de la imagen atiende específicamente cada token de la respuesta generada, no solo la atención interna del ViT. Se dejó fuera de este notebook por simplicidad de implementación inicial (requiere identificar y "enganchar" las capas de atención cruzada del decodificador, además de las del encoder visual), pero es la extensión más directa de este experimento si sus resultados iniciales son prometedores.

## 9. Próximos pasos sugeridos

1. Validar visualmente los primeros heatmaps generados (¿la atención se concentra en zonas con figuras, o se dispersa de forma pareja / en el fondo?) antes de ampliar la muestra.
2. Una vez validado sobre la muestra de esta sección, ampliar `N_IMAGENES_POR_TIPO` y comparar sistemáticamente la métrica `concentracion_top25pct` entre los 4 tipos de pregunta y, opcionalmente, entre corridas con acierto vs. error del modelo.
3. Incorporar los heatmaps resultantes y los hallazgos a la sección 3.4 / Capítulo 4 de la Memoria -- incluyendo, si corresponde, una nota metodológica sobre por qué fue necesario un monkey-patch en vez de forward hooks (sección 4): es un hallazgo relevante sobre las limitaciones de explicabilidad en modelos con atención fusionada (SDPA), no solo un detalle de implementación.


## 10. (Opcional) revertir el patch

Si quieres volver a dejar moondream2 en su estado original (por ejemplo, para comparar velocidad/memoria con y sin la versión instrumentada, o antes de correr otra evaluación en la misma sesión de Colab), esta celda restaura la función `attn()` original.


In [ ]:
vision_mod.attn = _original_vision_attn
print("attn() original restaurada en vision.py.")
